# Обучение классификатора позиций чеков (итерация 1, baseline)

Ноутбук — только фасад: конфиг, вызовы `kkt_clf`, вывод. Запуск: **Run All**.

Пресет `default` — дефолты под ~6 млн строк, `ods_light` — быстрый прогон на ODS. `OVERRIDES` накладывается поверх пресета; итоговый конфиг печатается ниже и сохраняется в артефакт.

In [ ]:
import json
import sys

sys.path.insert(0, "..")

import pandas as pd
from IPython.display import display

from kkt_clf import build_config, plots, run_training, setup_logging, verify_roundtrip

setup_logging()
pd.set_option("display.max_colwidth", 80)

## Конфиг

In [ ]:
PRESET = "ods_light"  # "default" — под ~6 млн строк (см. README)

OVERRIDES = {
    "output_dir": "../outputs/train_ods",
    "seed": 42,
    "data": {
        "path": "../fixtures/ods.parquet",
        "max_rows": None,  # None — брать всё; иначе стратифицированный по L3 сэмпл
        "min_count": 10,
        "split": [0.8, 0.1, 0.1],
    },
    "features": {
        "char": {"ngram_range": [2, 5], "min_df": 2, "max_features": 50_000, "sublinear_tf": True},
        "word": {"ngram_range": [1, 1], "min_df": 1, "max_features": 20_000, "sublinear_tf": True},
        "vectorizer_fit_rows": None,  # подвыборка train для fit векторизаторов
    },
    "model": {"hidden": [256], "dropout": [0.3], "activation": "relu", "norm": None},
    "loss": {
        "head_weights": {"l1": 0.3, "l2": 0.3, "l3": 1.0},
        "class_weight_mode": {"l1": "none", "l2": "none", "l3": "none"},  # none | inv_freq
        "class_weight_alpha": 0.5,
    },
    "train": {
        "optimizer": "adamw", "lr": 1e-3, "weight_decay": 0.0,
        "batch_size": 256, "epochs": 15,
        "early_stopping_metric": "l3_macro_f1", "patience": 3,
        "device": "auto",
    },
    "reject": {
        "tau_mode": "fixed",  # fixed | target_precision
        "tau": 0.5,
        "target_precision": 0.95,
        "reject_on_mismatch_l2": False,
        "reject_on_mismatch_l1": False,
    },
}

cfg = build_config(PRESET, OVERRIDES)
print(cfg.to_json())

## Обучение

In [ ]:
result = run_training(cfg)

## Данные

In [ ]:
print(json.dumps(result.data.report, ensure_ascii=False, indent=2))
print("Исключённые редкие классы L3 (первые 20 из", len(result.data.rare_classes), "):")
display(result.data.rare_classes.head(20))
print("Повторяющиеся name:", result.data.duplicates.n_duplicate_rows)
display(result.data.duplicates.examples)

In [ ]:
print(json.dumps(result.feature_stats, ensure_ascii=False, indent=2))
print("Параметры модели по слоям:")
display(result.param_counts)

## Кривые обучения

In [ ]:
display(result.history)
print("Лучшая эпоха:", result.best_epoch)
fig = plots.plot_learning_curves(result.history)

## Отказ: выбор τ на val

In [ ]:
print(json.dumps(result.tau_selection, ensure_ascii=False, indent=2))
display(result.tau_table.iloc[::5])
fig = plots.plot_tau_curve(result.tau_table, result.tau_selection["tau"])

## Метрики на test

`no_reject` — без отказа: L1/L2 по своим головам (`head`) и по пути от предсказанного L3 (`from_l3`). `with_reject` — на принятых примерах. Полные `classification_report` — CSV в `reports/`.

In [ ]:
ev = result.evaluation
display(result.summary)
print(f"Покрытие (доля неотказанных) на test: {ev['coverage']:.4f} ({ev['n_accepted']} из {ev['n_rows']})")
print("Строк с пустым вектором признаков на test:", ev["n_empty_features"])
print("Классов L3 с support <= 2 на test (шумные метрики):", ev["n_noisy_l3_classes"])
print("Отчёты:", result.report_dir)

In [ ]:
display(ev["reports"]["no_reject__L1__head"])
display(ev["reports"]["no_reject__L2__head"].tail(5))
display(ev["reports"]["no_reject__L3__head"].tail(5))
display(ev["reports"]["with_reject__L3__head"].tail(5))

In [ ]:
fig = plots.plot_f1_vs_freq(ev["f1_vs_freq"])

## Ошибки и рассогласование голов

In [ ]:
display(ev["top_errors"].head(20))
display(ev["mismatch_rates"])

## Ограничения

In [ ]:
for line in result.limitations:
    print("-", line)

## Проверка round-trip артефакта

In [ ]:
print("Предсказания до сохранения и после загрузки совпадают:", verify_roundtrip(result))
print("Артефакт:", result.artifact_dir)